<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L07-slice-discovery/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L07-slice-discovery/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/document-intelligence/lessons/P02-L07-slice-discovery/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/document-intelligence/lessons/P02-L07-slice-discovery/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P02-L07 · Error analysis and slice discovery: is that regression real?

**You will build:** a slice finder over document attributes, a vectorised paired bootstrap
for the change in each slice's macro F1 between two model versions, Holm and
Benjamini-Hochberg corrections, the sample size a slice needs before it may block a release,
and the release decision itself.

**Time:** ~75 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download, no model
API · **Prerequisites:** T00-L01 (the 8 GB track), P02-L01 (field extraction you can
measure).

A candidate extractor, v2, beats the production one, v1, on the evaluation set's macro F1.
Most release reviews stop there. This one asks where v2 got worse, and whether any of those
drops is real — because an aggregate can rise while a slice of documents falls, and because
a search over enough slices will always find some that fell by chance.

By the end you will be able to:

1. Implement a slice finder that enumerates every attribute value, and every cross of two,
   above a minimum size.
2. Implement a vectorised paired bootstrap for the change in a slice's macro F1, with a
   percentile interval and a one-sided p-value.
3. Implement Holm's step-down and Benjamini-Hochberg's step-up adjustments, and explain why
   a release gate uses the first.
4. Compute the sample size a slice needs before it may block a release, and measure what
   searching deeper does to it.
5. Write a release decision that names which slice regressions are real and which slices are
   too small to say — and measure how often the shortcuts block a release in which nothing
   changed.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import itertools
import math
import random
import re
import sys
import time
import traceback
from statistics import NormalDist
from typing import Callable, Mapping, NamedTuple, Sequence

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__)
print("two model versions, one evaluation set, and the question every release review should")
print("ask before it asks anything else: worse for whom, and is that real?\n")

# Module 1's schema, unchanged: six fields, each with the TYPE that decides how it is compared.
SCHEMA: dict[str, str] = {
    "invoice_id": "id",
    "invoice_date": "date",
    "total_amount": "money",
    "currency": "id",
    "counterparty": "text",
    "payment_terms_days": "integer",
}
FIELDS = tuple(SCHEMA)
MATCH_MODES = ("exact", "normalised")
MONTH_NAMES = ("January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December")
MONTH_INDEX = {name.lower(): i + 1 for i, name in enumerate(MONTH_NAMES)}
COMPANY_SUFFIXES = frozenset({
    "gmbh", "bv", "nv", "ag", "kg", "ltd", "limited", "inc", "incorporated",
    "plc", "llc", "sa", "sas", "srl", "spa", "oy", "ab", "as", "co",
})

# The document attributes a slice can be cut on, in the order every slice name uses them.
ATTRIBUTES = ("vendor", "layout", "language", "pages", "scan")

N_DOCS = 1600               # documents in the evaluation set
SEED = 20260934             # the evaluation set
EXTRACTOR_SEEDS = {"v1": 111, "v2": 211}
BOOT_SEED = 7               # every bootstrap below draws from this, per slice

# The release policy. These four numbers are DECISIONS your team writes down before it looks at
# the results, not facts about the world, and nothing here claims they are typical.
ALPHA = 0.05                # family-wise chance of blocking a release on a false alarm
MARGIN = 0.05               # a drop in a slice's macro F1 this large is worth blocking for
POWER = 0.80                # chance a slice big enough to block would catch a MARGIN drop
N_BOOT = 2000               # bootstrap replicates per slice
MIN_SLICE = 10              # below this a slice is not even estimated

VERDICTS = ("real regression", "too small to say", "no regression found")


class Counts(NamedTuple):
    """Per-document cell counts for one model: three int arrays of shape (documents, fields)."""
    tp: np.ndarray
    fp: np.ndarray
    fn: np.ndarray


class Slice(NamedTuple):
    """A named subset of the evaluation set: `index` holds document POSITIONS, ascending."""
    name: str
    index: np.ndarray


class Bootstrap(NamedTuple):
    """The paired bootstrap of (candidate minus champion) macro F1 on one set of documents."""
    delta: float            # point estimate on the documents as given
    lo: float               # percentile interval, lower end
    hi: float               # percentile interval, upper end
    p_value: float          # one-sided: small when the candidate is WORSE
    se: float               # standard deviation of the replicates
    replicates: np.ndarray  # shape (n_boot,)


class SliceResult(NamedTuple):
    """One slice's row in the analysis: its size and its bootstrap summary."""
    name: str
    n: int
    delta: float
    lo: float
    hi: float
    p_value: float
    se: float


class ReleaseDecision(NamedTuple):
    """The artefact: a verdict per slice, the Holm-adjusted p behind it, and the outcome."""
    verdicts: dict          # slice name -> one of VERDICTS, in the order the slices came in
    holm: dict              # slice name -> Holm-adjusted p-value
    blocking: tuple         # names whose verdict is "real regression", in that order
    too_small: tuple        # names whose verdict is "too small to say", in that order
    ship: bool              # True only when nothing blocks


def normal_quantile(q: float) -> float:
    """The standard normal quantile: normal_quantile(0.975) is about 1.96. Given to you."""
    return NormalDist().inv_cdf(q)


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("find_slices",),
    "exercise 2": ("macro_f1_from_counts",),
    "exercise 3": ("paired_bootstrap",),
    "exercise 4": ("holm_adjust", "bh_adjust"),
    "exercise 5": ("required_slice_size",),
    "exercise 6": ("release_decision",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (paired_bootstrap)"; several -> "exercises 1, 2 and 3"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The evaluation set, and the attributes you can slice it on

The documents are module 1's remittance advices, generated inside the lesson from a fixed
seed, so every student sees the same set. Each one now also carries the five attributes a
document pipeline usually knows before it extracts anything: who sent it, what its layout
family is, what language it is in, how many pages it runs to, and how it arrived — born
digital, scanned cleanly, scanned badly, or faxed.

Page count is a number, and a slice per exact page count would be a pile of slices too small
to measure, so `page_bucket` bins it before anything is sliced. Run the cell and read how
unevenly the documents fall: that unevenness is the whole problem of this lesson.

In [ ]:
VENDORS = (
    "Nordwind Logistik GmbH", "Vantor Marine B.V.", "Helix Pharma Limited",
    "Caldera Energy PLC", "Brightwater Analytics Ltd", "Orsini Costruzioni SRL",
    "Kestrel Freight Inc", "Aalto Terveys Oy", "Meridian Custody AG",
    "Sable & Finch LLP", "Dunbar Reinsurance Ltd", "Petrarca Chimica SpA",
    "Lindqvist Verkstad AB", "Hollandse Kaasunie N.V.", "Argent Clearing SA",
    "Torrent Robotics Inc", "Vesper Maritime AS", "Kaneko Precision KK",
)
# A few large senders and a long tail of small ones, as in any accounts-payable inbox.
VENDOR_WEIGHTS = (0.30, 0.20, 0.14, 0.06, 0.05, 0.045, 0.04, 0.035, 0.03, 0.025, 0.02, 0.018,
                  0.015, 0.012, 0.01, 0.009, 0.008, 0.008)
LAYOUTS = {"single-column": 0.50, "two-column": 0.30, "table-heavy": 0.20}
LANGUAGES = {"en": 0.42, "de": 0.26, "fr": 0.16, "nl": 0.13, "fi": 0.03}
SCAN_QUALITIES = {"born-digital": 0.43, "clean-scan": 0.29, "poor-scan": 0.15, "fax": 0.13}
_CURRENCIES = {"EUR": "€", "USD": "$", "GBP": "£"}


def page_bucket(pages: int) -> str:
    """Bin a page count so that a slice on it has documents in it: 1, 2-3, 4-7, 8+."""
    if pages <= 1:
        return "1"
    if pages <= 3:
        return "2-3"
    if pages <= 7:
        return "4-7"
    return "8+"


def _surface_date(year: int, month: int, day: int, style: int) -> str:
    if style == 0:
        return f"{year:04d}-{month:02d}-{day:02d}"
    if style == 1:
        return f"{day:02d}/{month:02d}/{year:04d}"       # day first: the set is European
    return f"{day} {MONTH_NAMES[month - 1]} {year}"


def _surface_money(cents: int, currency: str, style: int) -> str:
    whole, part = divmod(cents, 100)
    if style == 0:
        return f"{_CURRENCIES[currency]}{whole:,}.{part:02d}"
    if style == 1:
        return f"{whole}.{part:02d} {currency}"
    return f"{currency} {whole:,}.{part:02d}"


def _surface_id(year: int, serial: int, style: int) -> str:
    if style == 0:
        return f"INV-{year}-{serial:04d}"
    if style == 1:
        return f"INV/{year}/{serial:04d}"
    return f"Inv {year} {serial:04d}"


def _draw(rng: random.Random, weights: Mapping[str, float]) -> str:
    return rng.choices(tuple(weights), weights=tuple(weights.values()))[0]


def build_corpus(n_docs: int = N_DOCS, seed: int = SEED) -> list[dict]:
    """Module 1's documents, each with its attributes. Deterministic.

    Gold values are the surface strings an annotator typed, as in module 1; an empty string
    means the field is not on the page. `attrs` holds the five sliceable attributes, with the
    page count already binned by `page_bucket`.
    """
    rng = random.Random(seed)
    corpus = []
    for i in range(n_docs):
        year = rng.choice((2025, 2026))
        month, day, serial = rng.randint(1, 12), rng.randint(1, 28), rng.randint(1, 9999)
        currency = rng.choice(tuple(_CURRENCIES))
        cents = rng.randint(1_50, 480_000_00)
        vendor = rng.choices(VENDORS, weights=VENDOR_WEIGHTS)[0]
        terms = rng.choice(("14", "30", "30", "45", "60", "", ""))  # 2 in 7 say nothing
        d_style, m_style, i_style = rng.randrange(3), rng.randrange(3), rng.randrange(3)
        pages = 1
        while pages < 14 and rng.random() < 0.55:
            pages += 1
        attrs = {"vendor": vendor, "layout": _draw(rng, LAYOUTS),
                 "language": _draw(rng, LANGUAGES), "pages": page_bucket(pages),
                 "scan": _draw(rng, SCAN_QUALITIES)}
        gold = {"invoice_id": _surface_id(year, serial, i_style),
                "invoice_date": _surface_date(year, month, day, d_style),
                "total_amount": _surface_money(cents, currency, m_style),
                "currency": currency, "counterparty": vendor, "payment_terms_days": terms}
        corpus.append({"doc_id": f"DOC-{i:04d}", "page_count": pages, "attrs": attrs,
                       "gold": gold})
    return corpus


CORPUS = build_corpus()
print(f"{len(CORPUS)} documents, {len(SCHEMA)} fields each, {len(ATTRIBUTES)} attributes\n")
for _attr in ATTRIBUTES:
    _sizes: dict[str, int] = {}
    for _doc in CORPUS:
        _sizes[_doc["attrs"][_attr]] = _sizes.get(_doc["attrs"][_attr], 0) + 1
    _ordered = sorted(_sizes.items(), key=lambda kv: (-kv[1], kv[0]))
    print(f"{_attr:9s} {len(_ordered):2d} values, largest {_ordered[0][1]:4d} documents, "
          f"smallest {_ordered[-1][1]:3d} ({_ordered[-1][0]})")

## 2. Two versions of the extractor, scored with module 1's harness

`run_extractor` is module 1's stand-in extractor with one change: its error rate now depends
on the document's attributes and on which VERSION is running. v1 is in production. v2 is the
candidate. Both make the mistakes module 1 taught you to score — misses, wrong values,
invented payment terms, house-style reformatting — and some documents are hard for any
version, so the two versions' errors are correlated document by document.

`own_error_rate` is the ground truth a real release review never has. You may read it. Try
not to: the next hour is about finding what it hides from the data alone, and finding
nothing else.

The scoring cell below it is module 1's harness, copied in unchanged — normaliser, matcher,
per-field scorer, macro F1 — because a lesson here opens alone in an empty directory and
cannot import another lesson's files.

In [ ]:
SHARED_FAILURE = {"born-digital": 0.02, "clean-scan": 0.03, "poor-scan": 0.06, "fax": 0.08}
SCAN_PENALTY = {"born-digital": 0.00, "clean-scan": 0.02, "poor-scan": 0.05, "fax": 0.07}
RARE_TEMPLATES = frozenset(VENDORS[-5:])   # the five smallest senders use unusual templates
DIGIT_TYPES = frozenset({"money", "date", "id"})
SHARED_SEED = 424242                       # the documents' own difficulty, shared by versions


def own_error_rate(version: str, attrs: Mapping[str, str], field_type: str) -> float:
    """Chance that `version` gets a present field wrong on its own, beyond the shared failures."""
    rate = SCAN_PENALTY[attrs["scan"]] + (0.15 if attrs["vendor"] in RARE_TEMPLATES else 0.0)
    if version == "v1":
        return rate + 0.08 + (0.05 if attrs["layout"] == "two-column" else 0.0)
    if version == "v2":
        if attrs["scan"] == "fax" and field_type in DIGIT_TYPES:
            return rate + 0.33
        return rate + 0.05
    raise ValueError(f"unknown version {version!r}")


def _reformat(value: str, field_type: str) -> str:
    """The extractor's house style: same meaning, different surface."""
    if field_type == "date":
        m = re.match(r"^(\d{4})-(\d{2})-(\d{2})$", value)
        if m:
            return f"{int(m.group(3))}/{int(m.group(2))}/{m.group(1)}"
        m = re.match(r"^(\d{2})/(\d{2})/(\d{4})$", value)
        if m:
            return f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
        m = re.match(r"^(\d{1,2}) (\w+) (\d{4})$", value)
        if m:
            return f"{m.group(3)}-{MONTH_INDEX[m.group(2).lower()]:02d}-{int(m.group(1)):02d}"
        return value
    if field_type == "money":
        return re.sub(r"[^\d.]", "", value.replace(",", ""))
    if field_type == "id":
        return re.sub(r"[^A-Za-z0-9]", "", value).upper()
    if field_type == "text":
        return value.replace(" B.V.", "").replace(" GmbH", "").replace(" Ltd", "")
    if field_type == "integer":
        return f"net {value}" if value else value
    return value


def _corrupt(value: str, field_type: str, rng: random.Random) -> str:
    """A genuinely wrong answer: same shape, different meaning."""
    if field_type == "money":
        digits = [i for i, c in enumerate(value) if c.isdigit()]
        i = rng.choice(digits)
        chars = list(value)
        chars[i] = rng.choice([d for d in "0123456789" if d != chars[i]])
        return "".join(chars)
    if field_type in {"integer", "id"}:
        digits = [i for i, c in enumerate(value) if c.isdigit()]
        if len(digits) >= 2:
            i, j = rng.sample(digits, 2)
            chars = list(value)
            chars[i], chars[j] = chars[j], chars[i]
            if "".join(chars) != value:
                return "".join(chars)
        return value + "1"
    if field_type == "date":
        m = re.search(r"\b(\d{1,2})\b", value)
        if m:
            bumped = str((int(m.group(1)) % 28) + 1).zfill(len(m.group(1)))
            return value[:m.start(1)] + bumped + value[m.end(1):]
        return value
    return rng.choice([c for c in VENDORS if c != value])


def run_extractor(corpus: Sequence[Mapping], version: str, seed: int) -> list[dict]:
    """Module 1's record format for every document: doc_id, gold, pred, conf. Deterministic.

    Two random streams per document. `shared` is the document's own difficulty and is the same
    for every version and every seed: a cell it fails, every version fails the same way. `own`
    is this run's, drawn from `seed`: a retrain with another seed makes different mistakes at
    the same rates.
    """
    records = []
    for i, doc in enumerate(corpus):
        shared = random.Random(SHARED_SEED * 1_000_003 + i)
        own = random.Random(seed * 1_000_003 + i)
        pred, conf = {}, {}
        for field, field_type in SCHEMA.items():
            truth = doc["gold"][field]
            hardness, roll = shared.random(), own.random()
            if not truth:                              # absent: invent terms, or stay quiet
                if roll < 0.20:
                    pred[field] = own.choice(("30", "14", "60"))
                    conf[field] = round(own.uniform(0.30, 0.72), 3)
                else:
                    pred[field], conf[field] = "", round(own.uniform(0.80, 0.98), 3)
                continue
            if hardness < SHARED_FAILURE[doc["attrs"]["scan"]]:
                source = shared                        # hard for every version
            elif roll < own_error_rate(version, doc["attrs"], field_type):
                source = own                           # this version's own mistake
            else:
                source = None
            if source is not None:
                if source.random() < 0.4:
                    pred[field], conf[field] = "", 0.0
                else:
                    pred[field] = _corrupt(truth, field_type, source)
                    conf[field] = round(own.uniform(0.34, 0.86), 3)
            elif own.random() < 0.7:
                pred[field] = _reformat(truth, field_type)
                conf[field] = round(own.uniform(0.62, 0.95), 3)
            else:
                pred[field], conf[field] = truth, round(own.uniform(0.86, 0.99), 3)
        records.append({"doc_id": doc["doc_id"], "gold": dict(doc["gold"]),
                        "pred": pred, "conf": conf})
    return records

In [ ]:
# Module 1's harness, unchanged apart from the fuzzy mode module 1 talked you out of.
class FieldScore(NamedTuple):
    """What one field scored over the whole corpus, under one matching mode."""
    field: str
    mode: str
    tp: int
    fp: int
    fn: int
    precision: float
    recall: float
    f1: float


def normalise_value(value: str, field_type: str) -> str:
    """Module 1's normaliser, unchanged."""
    if not isinstance(value, str) or not value.strip():
        return ""
    if field_type not in set(SCHEMA.values()):
        raise ValueError(f"unknown field_type {field_type!r}")
    raw = " ".join(value.split())
    fallback = raw.upper()
    if field_type == "money":
        body = re.sub(r"[^0-9.,-]", "", raw)
        last_comma, last_dot = body.rfind(","), body.rfind(".")
        if last_comma >= 0 and last_dot >= 0:
            if last_comma > last_dot:
                body = body.replace(".", "").replace(",", ".")
            else:
                body = body.replace(",", "")
        elif last_comma >= 0:
            body = body.replace(",", ".") if re.search(r",\d{1,2}$", body) \
                else body.replace(",", "")
        try:
            return f"{float(body):.2f}"
        except ValueError:
            return fallback
    if field_type == "date":
        m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})$", raw)
        if m:
            y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
        else:
            m = re.match(r"^(\d{1,2})[/.-](\d{1,2})[/.-](\d{4})$", raw)
            if m:
                d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
            else:
                m = re.match(r"^(\d{1,2})\s+([A-Za-z]+),?\s+(\d{4})$", raw)
                if m and m.group(2).lower() in MONTH_INDEX:
                    d, mo, y = int(m.group(1)), MONTH_INDEX[m.group(2).lower()], int(m.group(3))
                else:
                    return fallback
        if not (1 <= mo <= 12 and 1 <= d <= 31):
            return fallback
        return f"{y:04d}-{mo:02d}-{d:02d}"
    if field_type == "integer":
        m = re.search(r"\d+", raw)
        return str(int(m.group(0))) if m else fallback
    if field_type == "id":
        stripped = re.sub(r"[^A-Za-z0-9]", "", raw)
        return stripped.upper() if stripped else fallback
    lowered = raw.lower().replace(".", "").replace(",", "")
    tokens = re.sub(r"[^a-z0-9]+", " ", lowered).split()
    while tokens and tokens[-1] in COMPANY_SUFFIXES:
        tokens.pop()
    return " ".join(tokens)


def match_value(pred: str, gold: str, field_type: str, mode: str) -> bool:
    """Module 1's matcher: an empty side never matches; exact or normalised equality."""
    if mode not in MATCH_MODES:
        raise ValueError(f"unknown mode {mode!r}; expected one of {list(MATCH_MODES)}")
    if not pred.strip() or not gold.strip():
        return False
    if mode == "exact":
        return pred == gold
    np_, ng_ = normalise_value(pred, field_type), normalise_value(gold, field_type)
    return bool(np_) and bool(ng_) and np_ == ng_


def score_field(records: Sequence[Mapping], field: str, mode: str) -> FieldScore:
    """Module 1's scorer: a wrong value is a false positive AND a false negative."""
    field_type = SCHEMA[field]
    tp = fp = fn = 0
    for record in records:
        pred, gold = record["pred"][field], record["gold"][field]
        has_pred, has_gold = bool(pred.strip()), bool(gold.strip())
        if has_pred and has_gold and match_value(pred, gold, field_type, mode):
            tp += 1
            continue
        if has_pred:
            fp += 1
        if has_gold:
            fn += 1
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return FieldScore(field, mode, tp, fp, fn, precision, recall, f1)


def macro_f1(records: Sequence[Mapping], mode: str) -> float:
    """Module 1's headline: the unweighted mean of the per-field F1 over every field."""
    if not SCHEMA:
        return 0.0
    return sum(score_field(records, f, mode).f1 for f in SCHEMA) / len(SCHEMA)


RECORDS = {v: run_extractor(CORPUS, v, s) for v, s in EXTRACTOR_SEEDS.items()}
_f1 = {v: macro_f1(RECORDS[v], "normalised") for v in RECORDS}
print(f"macro F1 on all {len(CORPUS)} documents (module 1's harness, normalised mode):")
for _v in RECORDS:
    print(f"  {_v}  {_f1[_v]:.3f}")
print(f"  change, v2 minus v1: {_f1['v2'] - _f1['v1']:+.3f}")
print("\nThe headline says ship. Whether a slice of these documents disagrees is what the rest")
print("of this lesson finds out.")

## 3. Exercise 1 — `find_slices`

A slice is every document that shares an attribute value: `vendor=Kestrel Freight Inc`,
`scan=fax`. Automatic slice finding enumerates them instead of waiting for someone to guess
which one to look at. The Slice Finder work frames model validation around this: overall
performance "can fail to reflect that of the smaller subsets" (Chung et al., arXiv 1807.06068).
Depth 2 crosses two attributes — `layout=two-column & scan=fax` — and finds regressions no
single attribute isolates, at a price section 7 measures.

<details><summary>💡 Hint 1 — what to think about</summary>

Three things make the output usable later: the ORDER is fixed (so two runs name the same
slices in the same order), the index holds positions in `docs` (so it can pick rows out of
an array), and a slice below `min_size` is not returned at all. A cross only exists when some
document has both values; never return an empty slice.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

For each depth from 1 to `max_order`, walk the combinations of that many attributes in
ATTRIBUTES order. For each combination, group the document positions by their tuple of
values; visit the groups in sorted order of that tuple; keep a group when it has at least
`min_size` members. Name it by joining `attribute=value` with ` & `, attributes in ATTRIBUTES
order.

</details>

In [ ]:
def find_slices(docs: Sequence[Mapping], min_size: int = MIN_SLICE,
                max_order: int = 1) -> list[Slice]:
    """Every attribute-value slice of `docs` with at least `min_size` documents.

    * Depth 1: one slice per value of each attribute, attributes in ``ATTRIBUTES`` order and
      values within an attribute in sorted order. Name: ``"scan=fax"``.
    * Depth 2 (when ``max_order >= 2``): one slice per pair of values of two different
      attributes that some document actually has, attribute pairs in ``ATTRIBUTES`` order,
      value pairs in sorted order. Name: ``"layout=two-column & scan=fax"`` — the attributes
      always in ``ATTRIBUTES`` order, never the other way round.
    * All depth-1 slices come before any depth-2 slice.
    * ``index`` is a sorted ``int64`` array of POSITIONS in `docs`, not doc ids.
    * A slice with fewer than `min_size` documents is dropped; exactly `min_size` is kept.
      An empty slice is never returned. ``max_order < 1`` raises ``ValueError``.

    Example:
        >>> docs = [{"attrs": {"vendor": "B", "layout": "x", "language": "en",
        ...                    "pages": "1", "scan": "fax"}},
        ...         {"attrs": {"vendor": "A", "layout": "x", "language": "en",
        ...                    "pages": "1", "scan": "fax"}}]
        >>> [(s.name, s.index.tolist()) for s in find_slices(docs, min_size=1)][:3]
        [('vendor=A', [1]), ('vendor=B', [0]), ('layout=x', [0, 1])]

    Returns:
        A list of ``Slice(name, index)``, in the order above.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_find_slices() -> None:
    docs = [{"attrs": {"vendor": v, "layout": l, "language": "en", "pages": "1", "scan": s}}
            for v, l, s in (("B", "x", "fax"), ("A", "y", "fax"), ("B", "y", "clean"),
                            ("A", "y", "fax"))]
    got = [(s.name, s.index.tolist()) for s in find_slices(docs, min_size=1)]
    assert got[:4] == [("vendor=A", [1, 3]), ("vendor=B", [0, 2]), ("layout=x", [0]),
                       ("layout=y", [1, 2, 3])], (
        f"depth 1 came back as {got[:4]} — attributes in ATTRIBUTES order, values sorted, "
        "and each index the POSITIONS of the documents in the list, ascending")
    assert all(s.index.dtype.kind == "i" for s in find_slices(docs, min_size=1)), \
        "index must be an integer array of positions, not doc ids or booleans"
    sizes = [len(s.index) for s in find_slices(docs, min_size=2)]
    assert 2 in sizes and min(sizes) >= 2, (
        f"min_size=2 kept sizes {sizes}: a slice with exactly min_size documents is KEPT, "
        "and one below it is dropped")
    deep = [s.name for s in find_slices(docs, min_size=1, max_order=2)]
    assert "vendor=A & layout=y" in deep and "layout=y & vendor=A" not in deep, (
        "depth-2 names put the two attributes in ATTRIBUTES order: 'vendor=A & layout=y'")
    assert "vendor=A & layout=x" not in deep, \
        "no document is both vendor=A and layout=x, so that cross must not be returned"
    assert deep.index("scan=fax") < deep.index("vendor=A & layout=y"), \
        "every depth-1 slice comes before any depth-2 slice"
    for attr in ATTRIBUTES:
        total = sum(len(s.index) for s in find_slices(CORPUS, min_size=1)
                    if s.name.startswith(attr + "="))
        assert total == len(CORPUS), (
            f"the {attr} slices at min_size=1 cover {total} documents, not {len(CORPUS)}: "
            "every document has exactly one value of each attribute")
    print("exercise 1 looks right")


_try("exercise 1", _check_find_slices)

Run this to see the search you are about to run, and the one you are not.

In [ ]:
def _show_slices() -> None:
    shallow = find_slices(CORPUS, MIN_SLICE, max_order=1)
    deep = find_slices(CORPUS, MIN_SLICE, max_order=2)
    sizes = sorted(len(s.index) for s in shallow)
    print(f"depth 1: {len(shallow)} slices of at least {MIN_SLICE} documents, "
          f"from {sizes[0]} to {sizes[-1]} documents each")
    print(f"depth 2: {len(deep)} slices — {len(deep) - len(shallow)} crosses on top")
    print("\nthe five smallest depth-1 slices:")
    for s in sorted(shallow, key=lambda s: (len(s.index), s.name))[:5]:
        print(f"  {s.name:32s} {len(s.index):4d} documents")


_try("slice search", _show_slices, needs=("exercise 1",))

## 4. Exercise 2 — macro F1 from counts, for every replicate at once

A bootstrap re-scores a slice thousands of times, so module 1's record-by-record scorer is
too slow to sit inside it. `cell_counts` (given, below) applies module 1's rule once per cell
and keeps the answer as three integer arrays of shape (documents, fields): true positives,
false positives, false negatives — a wrong value still counts as one of each. Sum any subset
of rows and you have that subset's per-field counts.

Your job is the scorer that turns summed counts into macro F1, for one set of totals or for a
whole stack of them: the field axis is always the LAST one, and whatever axes sit in front of
it survive into the answer.

<details><summary>💡 Hint 1 — what to think about</summary>

Module 1's F1 is 0.0 when a field has no true positives, whatever else happened, and when it
has no data at all. A division that produces a NaN or a warning inside a bootstrap poisons
every replicate that touches it, so guard it rather than clean up afterwards. And macro means
averaging F1 over fields, not pooling the counts first.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Precision and recall cancel down: F1 is twice the true positives over twice the true
positives plus the false positives plus the false negatives. Compute that per field with a
division that writes 0.0 wherever the denominator is zero, then take the mean over the last
axis.

</details>

In [ ]:
def cell_counts(records: Sequence[Mapping]) -> Counts:
    """Module 1's per-cell rule, kept per document: a match is a TP, and otherwise a present
    prediction is an FP and a present gold value an FN — both, for a wrong value."""
    shape = (len(records), len(SCHEMA))
    tp, fp, fn = (np.zeros(shape, dtype=np.int64) for _ in range(3))
    for i, record in enumerate(records):
        for k, (field, field_type) in enumerate(SCHEMA.items()):
            pred, gold = record["pred"][field], record["gold"][field]
            has_pred, has_gold = bool(pred.strip()), bool(gold.strip())
            if has_pred and has_gold and match_value(pred, gold, field_type, "normalised"):
                tp[i, k] = 1
                continue
            fp[i, k], fn[i, k] = has_pred, has_gold
    return Counts(tp, fp, fn)


def subset(counts: Counts, index: np.ndarray) -> Counts:
    """The rows of `counts` for the documents at `index`."""
    return Counts(counts.tp[index], counts.fp[index], counts.fn[index])


COUNTS = {v: cell_counts(RECORDS[v]) for v in RECORDS}


def macro_f1_from_counts(tp: np.ndarray, fp: np.ndarray, fn: np.ndarray) -> np.ndarray:
    """Macro F1 from summed per-field counts, vectorised over any leading axes.

    `tp`, `fp` and `fn` share one shape, with the FIELD axis last: ``(fields,)`` for one set
    of totals, ``(n_boot, fields)`` for a stack of bootstrap replicates. Per field, F1 is
    module 1's harmonic mean of precision and recall, 0.0 when the field has no true positives
    — including a field with no data at all. Macro F1 is the unweighted mean over the field
    axis. No NaN, no RuntimeWarning, no Python loop over the leading axes.

    Example:
        >>> macro_f1_from_counts(np.array([1, 0]), np.array([0, 0]), np.array([1, 0]))
        array(0.33333333)

    Returns:
        A float array of shape ``tp.shape[:-1]`` — a 0-d array for one set of totals.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_macro_f1() -> None:
    import warnings
    totals = [c.sum(axis=0) for c in COUNTS["v1"]]
    got, want = float(macro_f1_from_counts(*totals)), macro_f1(RECORDS["v1"], "normalised")
    assert abs(got - want) < 1e-12, (
        f"on v1's summed counts you gave {got:.6f}; module 1's macro_f1 on the same records "
        f"gives {want:.6f} — per-field F1 first, then the plain mean over fields")
    tp, fp, fn = np.array([4, 0]), np.array([0, 0]), np.array([0, 4])
    micro = 2 * 4 / (2 * 4 + 0 + 4)
    got = float(macro_f1_from_counts(tp, fp, fn))
    assert abs(got - 0.5) < 1e-12, (
        f"fields scoring F1 1.0 and 0.0 average to 0.5, you gave {got:.4f}"
        + (" — that is the POOLED (micro) F1; average the per-field F1s instead"
           if abs(got - micro) < 1e-9 else ""))
    stack = macro_f1_from_counts(np.array([[1, 1], [0, 1]]), np.zeros((2, 2)),
                                 np.array([[0, 1], [1, 1]]))
    assert np.shape(stack) == (2,), (
        f"a (2, fields) stack must give shape (2,), got {np.shape(stack)} — take the mean over "
        "the LAST axis only")
    with warnings.catch_warnings():
        warnings.simplefilter("error")
        zero = float(macro_f1_from_counts(np.zeros(3), np.zeros(3), np.zeros(3)))
    assert zero == 0.0, "a field with no data scores 0.0, as module 1 said; guard the division"
    print("exercise 2 looks right")


_try("exercise 2", _check_macro_f1)

The point-estimate reading: every depth-1 slice, v2 minus v1, sorted by how far it fell. This
is the table most release reviews are decided on.

In [ ]:
def _slice_delta(index: np.ndarray) -> float:
    a, b = subset(COUNTS["v1"], index), subset(COUNTS["v2"], index)
    f_a = macro_f1_from_counts(*(x.sum(axis=0) for x in a))
    f_b = macro_f1_from_counts(*(x.sum(axis=0) for x in b))
    return float(f_b - f_a)


def _show_point_estimates() -> None:
    rows = [(s.name, len(s.index), _slice_delta(s.index))
            for s in find_slices(CORPUS, MIN_SLICE)]
    rows.sort(key=lambda r: (r[2], r[0]))
    print(f"{'slice':34s}{'docs':>6s}{'v2 - v1':>10s}")
    for name, n, delta in rows[:10]:
        print(f"{name:34s}{n:6d}{delta:+10.3f}")
    fell = [r for r in rows if r[2] < 0]
    big = [r for r in rows if r[2] < -MARGIN]
    print(f"... {len(rows) - 10} more\n")
    print(f"{len(fell)} of {len(rows)} slices fell; {len(big)} fell by more than MARGIN "
          f"({MARGIN}): {', '.join(r[0] for r in big)}.")
    print("Which of those are real? Nothing in this table can tell you.")


_try("point estimates", _show_point_estimates, needs=("exercise 1", "exercise 2"))

## 5. Exercise 3 — `paired_bootstrap`

The bootstrap estimates a statistic's sampling distribution from the one sample you have, by
resampling it with replacement (Efron, 1979). Here the statistic is the change in macro F1,
candidate minus champion, and the unit you resample is the DOCUMENT: a replicate draws the
slice's documents with replacement and scores both versions on the same draw. Pairing is the
point — a document hard for v1 is usually hard for v2, and resampling the two separately
would count that shared difficulty as noise.

Report the percentile interval, and a one-sided p-value for "the candidate is worse": the
share of replicates at or above zero, counted as `(r + 1) / (n_boot + 1)` — the form North,
Curtis and Sham give for Monte Carlo p-values, which can never be exactly zero.

<details><summary>💡 Hint 1 — what to think about</summary>

One index matrix, used for both versions — draw it twice and the pairing is gone. The point
estimate is the change on the documents as given, not the mean of the replicates. Ties at
exactly zero count AGAINST a regression. And there should be no Python loop over replicates.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Draw a (n_boot, n) matrix of positions in one call. Indexing each count array with it and
summing over the document axis gives every replicate's per-field totals at once; feed those
to your exercise-2 scorer for each version and subtract. Cheaper still: count how often each
document was drawn in each replicate, and one matrix product with the per-document counts
gives the same totals. Then two quantiles, one count, one standard deviation.

</details>

In [ ]:
def paired_bootstrap(a: Counts, b: Counts, n_boot: int, rng: np.random.Generator,
                     alpha: float = ALPHA) -> Bootstrap:
    """Paired bootstrap of macro F1 of `b` (candidate) minus `a` (champion).

    `a` and `b` hold the SAME documents in the same order. Draw the resample as ONE index
    matrix, ``rng.integers(0, n, size=(n_boot, n))``, and score both versions on it.

    * ``delta``      macro F1 of `b` minus that of `a`, on the documents as given
    * ``replicates`` the same difference on each resample, shape ``(n_boot,)``
    * ``lo, hi``     the ``alpha/2`` and ``1 - alpha/2`` quantiles of the replicates
      (``np.quantile``'s default method): a 95% interval for ``alpha = 0.05``
    * ``p_value``    ``(1 + number of replicates >= 0) / (n_boot + 1)`` — small when the
      candidate is worse, 1.0 when the two versions are identical
    * ``se``         the standard deviation of the replicates, ``ddof=1``

    Different shapes for `a` and `b` raise ``ValueError``: they cannot be paired.

    Example:
        >>> ones, zeros = np.ones((3, 1), int), np.zeros((3, 1), int)
        >>> boot = paired_bootstrap(Counts(ones, zeros, zeros), Counts(ones, zeros, zeros),
        ...                         5, np.random.default_rng(0))
        >>> boot.delta, boot.p_value, boot.replicates.tolist()
        (0.0, 1.0, [0.0, 0.0, 0.0, 0.0, 0.0])

    Returns:
        ``Bootstrap(delta, lo, hi, p_value, se, replicates)``.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_bootstrap() -> None:
    same = subset(COUNTS["v1"], np.arange(40))
    twin = paired_bootstrap(same, same, 300, np.random.default_rng(1))
    assert np.all(twin.replicates == 0.0) and twin.p_value == 1.0, (
        "a version compared with ITSELF must give every replicate exactly 0 and p = 1.0 — "
        "if not, you resampled the two versions with different indices, or counted ties at "
        "zero as evidence of a regression")
    worse = Counts(same.tp * 0, same.fp + same.tp, same.fn + same.tp)   # v2 loses every hit
    down = paired_bootstrap(same, worse, 300, np.random.default_rng(2))
    assert down.delta < 0, "delta is the CANDIDATE (b) minus the champion (a)"
    assert down.p_value == 1 / 301, (
        f"every replicate is below zero, so p must be 1/(n_boot + 1) = {1 / 301:.5f}, got "
        f"{down.p_value:.5f} — the +1 on top and bottom keeps a p-value from being 0")
    index = np.arange(120)
    a, b = subset(COUNTS["v1"], index), subset(COUNTS["v2"], index)
    boot = paired_bootstrap(a, b, 500, np.random.default_rng(3))
    point = float(macro_f1_from_counts(*(x.sum(axis=0) for x in b))
                  - macro_f1_from_counts(*(x.sum(axis=0) for x in a)))
    assert abs(boot.delta - point) < 1e-12, (
        f"delta should be the change on the documents as given ({point:+.5f}), got "
        f"{boot.delta:+.5f} — not the mean of the replicates")
    assert np.shape(boot.replicates) == (500,), "one replicate per row of the index matrix"
    lo, hi = np.quantile(boot.replicates, [ALPHA / 2, 1 - ALPHA / 2])
    assert abs(boot.lo - lo) < 1e-12 and abs(boot.hi - hi) < 1e-12, (
        "lo and hi are the alpha/2 and 1 - alpha/2 quantiles of YOUR replicates — for a 95% "
        "interval the 2.5th and 97.5th percentiles, not the 5th and 95th")
    print("exercise 3 looks right")


_try("exercise 3", _check_bootstrap)

Run the bootstrap over every depth-1 slice. `evaluate_slices` (given) calls your function once
per slice, each with its own seeded generator, and times the whole sweep.

In [ ]:
def evaluate_slices(slices: Sequence[Slice], a: Counts, b: Counts, n_boot: int = N_BOOT,
                    seed: int = BOOT_SEED) -> list[SliceResult]:
    """One SliceResult per slice, from `paired_bootstrap` with a generator seeded per slice."""
    results = []
    for k, s in enumerate(slices):
        boot = paired_bootstrap(subset(a, s.index), subset(b, s.index), n_boot,
                                np.random.default_rng((seed, k)))
        results.append(SliceResult(s.name, len(s.index), boot.delta, boot.lo, boot.hi,
                                   boot.p_value, boot.se))
    return results


_CACHE: dict = {}


def _analysis() -> dict:
    """The depth-1 analysis of v2 against v1, recomputed whenever you redefine an exercise."""
    # The key holds the functions themselves, not their id()s: a redefined function can be
    # given the id of one that was freed, and an id key would then serve stale results.
    key = (find_slices, macro_f1_from_counts, paired_bootstrap)
    if _CACHE.get("key") != key:
        slices = find_slices(CORPUS, MIN_SLICE)
        t0 = time.perf_counter()
        results = evaluate_slices(slices, COUNTS["v1"], COUNTS["v2"])
        seconds = time.perf_counter() - t0
        overall = paired_bootstrap(COUNTS["v1"], COUNTS["v2"], N_BOOT,
                                   np.random.default_rng((BOOT_SEED, 10_000)))
        _CACHE.clear()
        _CACHE.update(key=key, slices=slices, results=results, seconds=seconds,
                      overall=overall)
    return _CACHE


def _show_intervals() -> None:
    run = _analysis()
    results, overall = run["results"], run["overall"]
    print(f"{len(results)} slices x {N_BOOT} paired replicates in {run['seconds']:.3f} s "
          f"(≈{len(results) * N_BOOT / run['seconds']:,.0f} slice-replicates per second)\n")
    print(f"whole evaluation set: v2 - v1 = {overall.delta:+.3f}, "
          f"95% interval [{overall.lo:+.3f}, {overall.hi:+.3f}]\n")
    print(f"{'slice':34s}{'docs':>6s}{'v2 - v1':>9s}{'95% interval':>20s}{'p':>9s}")
    for r in sorted(results, key=lambda r: (r.p_value, r.name))[:10]:
        print(f"{r.name:34s}{r.n:6d}{r.delta:+9.3f}   [{r.lo:+.3f}, {r.hi:+.3f}]"
              f"{r.p_value:9.4f}")
    print(f"... {len(results) - 10} more, all with larger p")
    # What pairing bought: the same bootstrap, but each version resampled on its own.
    a, b, n = COUNTS["v1"], COUNTS["v2"], len(CORPUS)
    rng = np.random.default_rng((BOOT_SEED, 20_000))

    def totals(counts: Counts) -> list[np.ndarray]:
        idx = rng.integers(0, n, size=(N_BOOT, n))
        drawn = np.bincount((idx + n * np.arange(N_BOOT)[:, None]).ravel(),
                            minlength=N_BOOT * n).reshape(N_BOOT, n)
        return [drawn @ x for x in counts]

    apart = macro_f1_from_counts(*totals(b)) - macro_f1_from_counts(*totals(a))
    print(f"\nwhole-set standard error, paired: {overall.se:.4f}; resampling the versions "
          f"separately: {apart.std(ddof=1):.4f}")
    print("The unpaired one counts each document's own difficulty as noise, twice.")


_try("slice intervals", _show_intervals, needs=("exercise 1", "exercise 2", "exercise 3"))

## 6. Exercise 4 — Holm and Benjamini-Hochberg

Every slice you searched is a test, and a test whose null is true still clears `ALPHA` with
probability `ALPHA`. Two corrections, two promises. Holm's procedure is sequentially
rejective: sort the p-values, reject the smallest if it clears `ALPHA / m`, the next if it
clears `ALPHA / (m-1)`, and stop at the first that fails — a step-down procedure that protects
against any false rejection, whichever nulls are true (Holm, 1979), and stays valid under any
dependence between the tests. Benjamini and Hochberg control the false discovery rate
instead — the expected share of false alarms among the rejections — with a step-up: find the
LARGEST `i` whose sorted p-value clears `i * ALPHA / m`, and reject it and everything smaller.

Both return ADJUSTED p-values in the order they came in: reject wherever the adjusted value is
at most `ALPHA`. Adjusted values are what a report can print.

<details><summary>💡 Hint 1 — what to think about</summary>

The difference between the two is which way the "stop" runs. In Holm, a failure early in the
sorted list must be able to veto everything after it. In BH, a success late in the list must
be able to rescue everything before it. Adjusted p-values express each as a running extreme.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Sort once and remember the order. Holm: multiply the i-th smallest (counting from 1) by
m - i + 1, take the running maximum from the smallest upwards, cap at 1. BH: multiply the
i-th smallest by m / i, take the running minimum from the LARGEST downwards, cap at 1. Then
put every value back where its p-value came from.

</details>

In [ ]:
def _valid_p(p_values: Sequence[float]) -> np.ndarray:
    """p-values as a 1-D float array, refusing anything that is not a probability. Given."""
    p = np.asarray(p_values, dtype=float).reshape(-1)
    if np.isnan(p).any() or (p < 0).any() or (p > 1).any():
        raise ValueError("p-values must lie in [0, 1]")
    return p


def holm_adjust(p_values: Sequence[float]) -> np.ndarray:
    """Holm step-down adjusted p-values, in the order of `p_values`.

    With the p-values sorted ascending, ``p_(1) <= ... <= p_(m)``, the adjusted value of
    ``p_(i)`` is ``min(1, max over j <= i of (m - j + 1) * p_(j))``. Reject where it is at
    most ALPHA. Worked example: ``(0.01, 0.04, 0.03, 0.045)`` sorts to ``0.01, 0.03, 0.04,
    0.045``; the multiplied values are ``0.04, 0.09, 0.08, 0.045``; the running maximum is
    ``0.04, 0.09, 0.09, 0.09`` — the last one is NOT rejected at 0.05, because Holm stopped at
    ``0.03``. An empty input gives an empty array; a value outside [0, 1] or NaN raises
    ``ValueError``.

    Example:
        >>> holm_adjust([0.01, 0.04, 0.03, 0.045]).round(3).tolist()
        [0.04, 0.09, 0.09, 0.09]

    Returns:
        A float array the length of `p_values`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def bh_adjust(p_values: Sequence[float]) -> np.ndarray:
    """Benjamini-Hochberg step-up adjusted p-values, in the order of `p_values`.

    With the p-values sorted ascending, the adjusted value of ``p_(i)`` is
    ``min(1, min over j >= i of m * p_(j) / j)``. Same worked example: the multiplied values
    are ``0.04, 0.06, 0.0533, 0.045``; the running minimum from the top is ``0.04, 0.045,
    0.045, 0.045`` — all four ARE rejected at 0.05, because the largest cleared its threshold
    and rescued the two before it. Empty in, empty out; invalid values raise ``ValueError``.

    Example:
        >>> bh_adjust([0.01, 0.04, 0.03, 0.045]).round(3).tolist()
        [0.04, 0.045, 0.045, 0.045]

    Returns:
        A float array the length of `p_values`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_adjust() -> None:
    p = [0.01, 0.04, 0.03, 0.045]
    holm = np.asarray(holm_adjust(p))
    assert holm.shape == (4,), "one adjusted value per p-value"
    assert np.allclose(holm, [0.04, 0.09, 0.09, 0.09]), (
        f"holm_adjust{tuple(p)} gave {holm.round(4).tolist()}, expected [0.04, 0.09, 0.09, "
        "0.09] — "
        + ("the last value is 0.045: you tested each p on its own threshold without the running "
           "maximum, so Holm did not stop at its first failure"
           if abs(holm[3] - 0.045) < 1e-9 else
           "multiply the i-th smallest by m - i + 1, take the running MAXIMUM from the "
           "smallest, and put each value back in its input position"))
    bh = np.asarray(bh_adjust(p))
    assert np.allclose(bh, [0.04, 0.045, 0.045, 0.045]), (
        f"bh_adjust{tuple(p)} gave {bh.round(4).tolist()}, expected [0.04, 0.045, 0.045, "
        "0.045] — multiply the i-th smallest by m / i and take the running MINIMUM from the "
        "LARGEST downwards, so a late success rescues the earlier p-values")
    assert np.allclose(holm_adjust([0.5, 0.9]), [1.0, 1.0]), \
        "adjusted p-values are probabilities: cap them at 1"
    assert np.asarray(holm_adjust([])).shape == (0,), "no p-values, no adjusted values"
    print("exercise 4 looks right")


_try("exercise 4", _check_adjust)

Apply both to the slice p-values, next to the uncorrected reading.

In [ ]:
def _show_corrections() -> None:
    results = _analysis()["results"]
    p = [r.p_value for r in results]
    holm, bh = holm_adjust(p), bh_adjust(p)
    m = len(results)
    print(f"{m} slices searched, so {m} tests\n")
    print(f"{'slice':34s}{'docs':>6s}{'raw p':>9s}{'Holm':>8s}{'BH':>8s}")
    for k in sorted(range(m), key=lambda k: (p[k], results[k].name))[:6]:
        print(f"{results[k].name:34s}{results[k].n:6d}{p[k]:9.4f}{holm[k]:8.3f}{bh[k]:8.3f}")
    for label, values in (("uncorrected", np.asarray(p)), ("Holm", holm), ("BH", bh)):
        hit = [results[k].name for k in range(m) if values[k] <= ALPHA]
        print(f"{label:>12s} at {ALPHA}: {len(hit)} slice(s) {', '.join(hit)}")
    print(f"\nA bootstrap p-value is never below 1/(N_BOOT + 1), so no Holm-adjusted p here can "
          f"be below {m}/({N_BOOT} + 1) = {m / (N_BOOT + 1):.4f}.")
    print("Section 7 shows what happens to that floor when the search goes deeper.")


_try("corrections", _show_corrections, needs=("exercise 1", "exercise 2", "exercise 3",
                                              "exercise 4"))

## 7. Exercise 5 — how big must a slice be before it may block a release?

A slice that cannot detect a regression of `MARGIN` should not be allowed to clear the
release, and — this is the policy you are writing — should not be allowed to block it either:
its verdict is "too small to say". How small is too small follows from the normal
approximation for a one-sided test (NIST/SEMATECH e-Handbook, section 7.2.2.2):

    n = ceil( ((z(1 - alpha) + z(power)) * sigma / MARGIN) ** 2 )

where `z` is `normal_quantile`, `alpha` is the level the slice will actually be tested at,
and `sigma` is the per-document standard deviation of the change. You do not know `sigma`;
the handbook's advice is the best estimate available from a previous experiment. Yours is the
whole-set bootstrap: its standard error shrinks like one over the square root of the number of
documents, so `sigma` is that standard error times the square root of `n_ref`.

<details><summary>💡 Hint 1 — what to think about</summary>

Two normal quantiles, not one: the critical value alone gives a test that catches a
`MARGIN` drop only about half the time. The test is one-sided. The standard error scales with
one over root n, so it is the VARIANCE that scales with n. And a size is a whole number of
documents that must be enough, so round in the safe direction.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the inputs. Turn the reference standard error into a per-document standard
deviation using the reference size, add the two quantiles, scale by sigma over the margin,
square, round up.

</details>

In [ ]:
def required_slice_size(se_ref: float, n_ref: int, margin: float = MARGIN,
                        alpha: float = ALPHA, power: float = POWER) -> int:
    """Documents a slice needs so a one-sided test at `alpha` detects a drop of `margin`
    with probability `power`, given a standard error `se_ref` measured on `n_ref` documents.

    ``sigma = se_ref * sqrt(n_ref)``, then
    ``n = ceil(((normal_quantile(1 - alpha) + normal_quantile(power)) * sigma / margin) ** 2)``.
    `se_ref`, `n_ref` and `margin` must be positive and `alpha` and `power` strictly between 0
    and 1, or ``ValueError``.

    Example:
        >>> required_slice_size(se_ref=0.004, n_ref=1600, margin=0.05, alpha=0.05, power=0.8)
        64

    Returns:
        The minimum number of documents, as an int.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_required() -> None:
    got = required_slice_size(se_ref=0.004, n_ref=1600, margin=0.05, alpha=0.05, power=0.8)
    critical_only = math.ceil((normal_quantile(0.95) * 0.16 / 0.05) ** 2)
    two_sided = math.ceil(((normal_quantile(0.975) + normal_quantile(0.8)) * 0.16 / 0.05) ** 2)
    assert isinstance(got, int), f"return an int, got {type(got).__name__}"
    assert got == 64, (
        f"the docstring's example should give 64, you gave {got}"
        + (" — that is the critical value alone; add normal_quantile(power) for the power term"
           if got == critical_only else
           " — that uses z(1 - alpha/2); this test is one-sided, use z(1 - alpha)"
           if got == two_sided else
           " — sigma is se_ref * sqrt(n_ref), and the whole bracket is squared, then rounded UP"))
    assert required_slice_size(0.004, 1600, 0.05, 0.05 / 30, 0.8) > got, \
        "a stricter alpha (a bigger search) must need MORE documents"
    assert required_slice_size(0.004, 400, 0.05, 0.05, 0.8) < got, (
        "the same standard error measured on FEWER documents means a smaller sigma, so fewer "
        "documents are needed — did you divide by sqrt(n_ref) instead of multiplying?")
    print("exercise 5 looks right")


_try("exercise 5", _check_required)

The size a slice needs on this evaluation set, tested at the level Holm's first step uses —
`ALPHA` divided by the number of slices — and what a deeper search would cost.

In [ ]:
def _gate(m: int) -> int:
    """n_required for a family of m slices: Holm's first, strictest step."""
    return required_slice_size(_analysis()["overall"].se, len(CORPUS), MARGIN, ALPHA / m, POWER)


def _show_required() -> None:
    run = _analysis()
    results, se = run["results"], run["overall"].se
    m = len(results)
    need = _gate(m)
    eligible = [r for r in results if r.n >= need]
    print(f"whole-set standard error {se:.4f} on {len(CORPUS)} documents: "
          f"sigma = {se * math.sqrt(len(CORPUS)):.3f}")
    print(f"a slice needs {need} documents to catch a {MARGIN} drop with power {POWER} at "
          f"{ALPHA}/{m}")
    print(f"{len(eligible)} of {m} slices are that large; {m - len(eligible)} are too small "
          "to say anything either way\n")
    spread = [r.se * math.sqrt(r.n) for r in eligible]
    print(f"sigma measured slice by slice, over the eligible ones: {min(spread):.3f} to "
          f"{max(spread):.3f}")
    print("One sigma for every slice is an approximation; that range is how rough it is.")
    print("\nwhat searching deeper would cost:")
    print(f"{'search':>9s}{'slices':>8s}{'n needed':>10s}{'eligible':>10s}"
          f"{'smallest Holm p':>17s}{'replicates needed':>19s}")
    for depth in (1, 2):
        family = find_slices(CORPUS, MIN_SLICE, max_order=depth)
        m_d, need_d = len(family), _gate(len(family))
        eligible_d = sum(len(s.index) >= need_d for s in family)
        floor = m_d / (N_BOOT + 1)
        print(f"{'depth ' + str(depth):>9s}{m_d:8d}{need_d:10d}{eligible_d:10d}{floor:17.4f}"
              f"{math.ceil(m_d / ALPHA) - 1:19d}")
    print(f"\nWith {N_BOOT} replicates, a search whose smallest possible Holm p is above ALPHA "
          "cannot block a release, however large a drop it finds.")


_try("required size", _show_required, needs=("exercise 1", "exercise 2", "exercise 3",
                                              "exercise 5"))

## 8. Exercise 6 — `release_decision`, the artefact

Three verdicts, decided in this order. A slice with fewer than `n_required` documents is
**too small to say**, whatever its p-value — the policy fixed before anyone looked. Otherwise
it is a **real regression** when its Holm-adjusted p-value is at most `alpha`, with the family
being EVERY slice you searched, the ones that improved included. Otherwise **no regression
found**. The release ships only when nothing is a real regression.

Holm, not BH, because a release gate's promise is about ANY false block, and because these
slices overlap — one document sits in a slice for each attribute — so the tests are dependent,
and Holm's guarantee does not care. BH's list is still worth printing: it is the error-analysis
queue for the team, not the gate.

<details><summary>💡 Hint 1 — what to think about</summary>

The attractive shortcuts are all wrong somewhere: blocking on a big point estimate, blocking
on a raw p-value, correcting over only the slices that fell, and letting a small slice block
because its p-value looks strong. Your function must be immune to all four.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Adjust every result's p-value with your `holm_adjust`, once, in the order given. Walk the
results in order, apply the three rules in the order above, and record each verdict and each
adjusted p under the slice's name. Collect the blocking and too-small names in order; the
release ships when there are no blocking names.

</details>

In [ ]:
def release_decision(results: Sequence[SliceResult], n_required: int,
                     alpha: float = ALPHA) -> ReleaseDecision:
    """The release decision over every slice in `results`.

    For each result, in order, with ``holm`` the Holm-adjusted p-values of ALL the results:

    1. ``n < n_required``        -> ``"too small to say"`` (may neither block nor clear)
    2. ``holm <= alpha``         -> ``"real regression"`` (blocks the release)
    3. otherwise                 -> ``"no regression found"``

    ``ship`` is True exactly when no slice is a real regression. The point estimate
    ``delta`` plays no part: a large drop with a large p-value is not evidence, and a small,
    precise drop is.

    Example:
        >>> rows = [SliceResult("scan=fax", 220, -0.12, -0.15, -0.09, 0.0005, 0.01),
        ...         SliceResult("vendor=Tiny", 12, -0.20, -0.40, 0.00, 0.0005, 0.1)]
        >>> d = release_decision(rows, n_required=150)
        >>> d.verdicts["scan=fax"], d.verdicts["vendor=Tiny"], d.ship
        ('real regression', 'too small to say', False)

    Returns:
        ``ReleaseDecision(verdicts, holm, blocking, too_small, ship)``.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_decision() -> None:
    def row(name: str, n: int, delta: float, p: float) -> SliceResult:
        return SliceResult(name, n, delta, delta - 0.02, delta + 0.02, p, 0.01)
    many = [row(f"quiet {k}", 500, 0.01, 0.8) for k in range(8)]
    d = release_decision([row("big drop", 400, -0.20, 0.30), row("precise", 900, -0.01, 0.001),
                          *many], n_required=200)
    assert d.verdicts["big drop"] == "no regression found", (
        "a -0.20 point estimate with p = 0.30 is not evidence of anything — the verdict "
        "must come from the adjusted p-value, never from delta")
    assert d.verdicts["precise"] == "real regression", \
        "p = 0.001 over 10 slices is 0.01 after Holm: a small drop, measured precisely, is real"
    d = release_decision([row("lucky", 400, -0.03, 0.01), *many], n_required=200)
    assert d.verdicts["lucky"] == "no regression found" and d.ship, (
        "p = 0.01 is significant on its own, but it was one of 9 slices searched: Holm makes "
        "it 0.09. Did you compare the RAW p-value with alpha?")
    d = release_decision([row("tiny", 12, -0.30, 0.0001), *many], n_required=200)
    assert d.verdicts["tiny"] == "too small to say" and d.too_small == ("tiny",) and d.ship, (
        "a slice below n_required is 'too small to say' however small its p-value, and it "
        "does not block the release")
    print("exercise 6 looks right")


_try("exercise 6", _check_decision)

The release note. Every figure in it is computed above; none is typed.

In [ ]:
def release_note() -> str:
    """The decision for v2, rendered for the people who have to act on it."""
    run = _analysis()
    results, overall = run["results"], run["overall"]
    by_name = {r.name: r for r in results}
    m = len(results)
    need = _gate(m)
    decision = release_decision(results, need)
    lines = [
        "RELEASE DECISION — candidate v2 against production v1",
        f"evaluation set  {len(CORPUS)} documents; {m} slices searched (one attribute each, "
        f"at least {MIN_SLICE} documents)",
        f"policy          block on a slice regression significant at family-wise {ALPHA} "
        f"(Holm),",
        f"                in a slice of at least {need} documents — enough to catch a {MARGIN} "
        f"drop in macro F1 with power {POWER}",
        f"whole set       v2 - v1 = {overall.delta:+.3f}, 95% interval "
        f"[{overall.lo:+.3f}, {overall.hi:+.3f}]",
        "",
        f"DECISION: {'SHIP' if decision.ship else 'DO NOT SHIP'} — "
        f"{len(decision.blocking)} real slice regression(s)",
        "",
        "real regressions (block the release):",
    ]
    for name in decision.blocking:
        r = by_name[name]
        lines.append(f"  {name:32s} {r.n:4d} docs  change {r.delta:+.3f}  "
                     f"95% [{r.lo:+.3f}, {r.hi:+.3f}]  Holm p {decision.holm[name]:.3f}")
    lines += ["", f"too small to say (fewer than {need} documents; may neither block nor clear):"]
    for name in decision.too_small:
        r = by_name[name]
        flag = "  <- Holm flags it; the size gate overrules" \
            if decision.holm[name] <= ALPHA else ""
        lines.append(f"  {name:32s} {r.n:4d} docs  change {r.delta:+.3f}  "
                     f"Holm p {decision.holm[name]:.3f}{flag}")
    cleared = [n for n, v in decision.verdicts.items() if v == "no regression found"]
    lines += ["", f"no regression found ({len(cleared)} slices): " + ", ".join(cleared)]
    return "\n".join(lines)


_try("release note", lambda: print(release_note()),
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5",
            "exercise 6"))

## 9. Five rules on this release, then on releases in which nothing changed

Your decision named some slices. Four shortcuts would have named others: block on any drop,
block on a drop bigger than `MARGIN`, block on an uncorrected p-value, block on Holm with no
size gate. Here they are on this release — and, because this is a generated evaluation set,
next to what no release review ever gets to see: the generator's EXPECTED change on each
slice's documents, averaged over fresh retrains of both versions, and each slice's share of
fax documents, which is where `own_error_rate` put the change in v2.

In [ ]:
TRUTH_RETRAINS = 12         # retrains of each version behind the "expected" column


def _rules(results: Sequence[SliceResult], need: int) -> dict[str, list[str]]:
    """The slices each rule would block, from the same results."""
    holm = holm_adjust([r.p_value for r in results])
    return {
        "1 any drop": [r.name for r in results if r.delta < 0],
        "2 drop > MARGIN": [r.name for r in results if r.delta < -MARGIN],
        "3 uncorrected p": [r.name for r in results if r.p_value <= ALPHA],
        "4 Holm, no size gate": [r.name for r, h in zip(results, holm) if h <= ALPHA],
        "5 your decision": list(release_decision(results, need).blocking),
    }


def _expected_changes(slices: Sequence[Slice]) -> dict[str, float]:
    """Macro F1 change per slice, averaged over TRUTH_RETRAINS retrains of each version."""
    total = np.zeros(len(slices))
    for r in range(TRUTH_RETRAINS):
        a = cell_counts(run_extractor(CORPUS, "v1", 5000 + r))
        b = cell_counts(run_extractor(CORPUS, "v2", 7000 + r))
        for k, s in enumerate(slices):
            total[k] += (macro_f1_from_counts(*(x[s.index].sum(axis=0) for x in b))
                         - macro_f1_from_counts(*(x[s.index].sum(axis=0) for x in a)))
    return {s.name: float(total[k] / TRUTH_RETRAINS) for k, s in enumerate(slices)}


def _show_this_release() -> None:
    run = _analysis()
    results, slices = run["results"], run["slices"]
    need = _gate(len(results))
    rules = _rules(results, need)
    expected = _expected_changes(slices)
    fax = np.array([doc["attrs"]["scan"] == "fax" for doc in CORPUS])
    share = {s.name: float(fax[s.index].mean()) for s in slices}
    print("this release — how many slices each rule would block:")
    for rule, blocked in rules.items():
        print(f"  rule {rule:22s} blocks {len(blocked):2d} slice(s)")
    print(f"\nevery slice a rule blocked, measured against the generator's expected change:")
    print(f"{'slice':34s}{'docs':>6s}{'fax':>6s}{'measured':>10s}{'expected':>10s}  blocked by")
    named = [r for r in results if any(r.name in b for b in rules.values())]
    for r in sorted(named, key=lambda r: (r.delta, r.name)):
        by = " ".join(rule.split()[0] for rule, b in rules.items() if r.name in b)
        print(f"{r.name:34s}{r.n:6d}{share[r.name]:6.0%}{r.delta:+10.3f}"
              f"{expected[r.name]:+10.3f}  rules {by}")
    for label, group in (("below the size gate", [r for r in results if r.n < need]),
                         ("at or above it", [r for r in results if r.n >= need])):
        gap = np.mean([abs(r.delta - expected[r.name]) for r in group])
        print(f"mean |measured - expected| over the {len(group)} slices {label}: {gap:.3f}")


_try("this release", _show_this_release,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5",
            "exercise 6"))

The measurement that matters is the next one. It retrains v1 with a fresh seed, twice, and
compares the two retrains exactly as you compared v2 with v1: the same recipe, so the right
answer is always "ship". Each rule is scored by how many of those null releases it would have
blocked. The size gate is the one you fixed on the real release; a policy is not re-tuned per
comparison.

One disclosure first. The seeds behind this evaluation set and its two versions were picked,
from a search over candidates, so that this one release shows every failure the lesson teaches
at least once, a tiny slice that Holm alone flags among them. The release above is a worked
example, not a typical one. The null releases re-run the whole pipeline on fresh retrains:
read them for how often each rule fails when nothing changed.

In [ ]:
NULL_RELEASES = 20


def _show_null_releases() -> None:
    run = _analysis()
    results, slices = run["results"], run["slices"]
    need = _gate(len(results))
    t0 = time.perf_counter()
    blocked_in = {rule: 0 for rule in _rules(results, need)}
    for r in range(NULL_RELEASES):
        a = cell_counts(run_extractor(CORPUS, "v1", 9000 + 2 * r))
        b = cell_counts(run_extractor(CORPUS, "v1", 9001 + 2 * r))
        null = evaluate_slices(slices, a, b, n_boot=1000, seed=100 + r)
        for rule, hit in _rules(null, need).items():
            blocked_in[rule] += bool(hit)
    print(f"{NULL_RELEASES} null releases (v1 retrained against v1 retrained, 1000 replicates "
          f"per slice), {time.perf_counter() - t0:.1f} s:")
    for rule, count in blocked_in.items():
        print(f"  rule {rule:22s} blocks {count:2d} of {NULL_RELEASES}")


_try("null releases", _show_null_releases,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5",
            "exercise 6"))

## 10. Common mistakes

- **Deciding on the point estimate.** A small slice's change swings widely from one sample
  to the next; section 9 sets each blocked slice's measured change beside its expected one.
- **Correcting over the slices that fell.** The family is every slice you searched. Filter to
  the drops first and you have quietly searched them all and then pretended you had not.
- **Resampling the two versions separately.** It throws away the pairing and inflates the
  interval; the whole-set standard errors in section 5 show by how much.
- **A Holm that never stops.** Testing each sorted p-value against its own threshold, without
  the running maximum, rejects hypotheses after an earlier one has failed.
- **BH as a release gate.** It controls the share of false alarms among the flags, not the
  chance of any false block, and its guarantee assumes the tests are independent or positively
  dependent. Use it for the error-analysis queue.
- **Letting a small slice block.** A small slice's measured change sits far from what the
  version really does to those documents; section 9 prints both, and the gap, by size.
- **Searching deeper without more replicates.** A bootstrap p-value cannot be below
  1/(N_BOOT + 1); past `ALPHA * (N_BOOT + 1)` slices, Holm can reject nothing at all.
- **A p-value of exactly zero.** Without the +1, one lucky run of replicates makes a
  regression look infinitely certain.

The fourth one is worth seeing rather than believing. Run this.

In [ ]:
def _show_holm_that_never_stops() -> None:
    p = np.array([0.01, 0.04, 0.03, 0.045])
    order = np.argsort(p, kind="stable")
    never_stops = np.zeros(p.size, dtype=bool)
    never_stops[order] = p[order] <= ALPHA / (p.size - np.arange(p.size))
    print(f"p-values                          {p.tolist()}")
    print(f"Holm, which stops at a failure    {(holm_adjust(p) <= ALPHA).tolist()}")
    print(f"a Holm that never stops           {never_stops.tolist()}")
    print(f"Benjamini-Hochberg                {(bh_adjust(p) <= ALPHA).tolist()}")
    print("The one that never stops rejects the largest p-value after the second-smallest "
          "failed: a rejection neither procedure makes for that reason.")


_try("a Holm that never stops", _show_holm_that_never_stops, needs=("exercise 4",))

## 11. Self-check

1. On the whole evaluation set, v2's macro F1 is higher than v1's and the interval excludes
   zero. What does that tell you about the fax slice?
   - (a) nothing on its own — an aggregate can rise while a slice falls, which is why the
         slices are searched at all
   - (b) that the fax slice cannot have fallen by more than the interval's width
   - (c) that any fax drop is noise, because the aggregate improved significantly

2. A small vendor slice has a Holm-adjusted p below `ALPHA`, but fewer documents than the
   size gate. The decision calls it "too small to say". The best justification is:
   - (a) Holm's procedure is not valid for slices of different sizes
   - (b) small vendors matter less to the business than large ones
   - (c) the policy, fixed before looking, says a slice this small can neither clear nor
         block, because its measured change is too far from the true one to act on —
         section 9 sets the two side by side

3. Holm flags one slice; Benjamini-Hochberg flags three. Which should gate the release?
   - (a) Holm: the gate must bound the chance of blocking on ANY false alarm, and it stays
         valid when overlapping slices make the tests dependent; BH's three are the
         error-analysis queue
   - (b) BH: it has more power, so it misses fewer real regressions
   - (c) whichever flags fewer slices, to be safe

4. A colleague's Holm sorts the p-values and rejects every one that clears its own threshold
   `ALPHA / (m - i + 1)`, without stopping at the first that fails. The consequence is:
   - (a) none — the thresholds only grow, so a later p-value can only clear its threshold if
         every earlier one did
   - (b) it can reject a hypothesis after an earlier, smaller p-value has failed, which the
         family-wise guarantee does not cover
   - (c) it becomes the Benjamini-Hochberg procedure

5. Section 7 shows that with `N_BOOT` replicates no depth-2 slice could block the release,
   however large its drop. Why?
   - (a) depth-2 slices are all below the size gate by construction
   - (b) Holm's procedure forbids overlapping slices
   - (c) a bootstrap p-value can never be below 1/(N_BOOT + 1), so once the number of slices
         times that floor exceeds `ALPHA`, no Holm-adjusted p-value can reach `ALPHA`

Answers come with this lesson's worked solution when you enrol on Synapsa.

One last cell: the decision in five lines, every one of them computed — the summary a
release board will ask you for before it reads anything else.

In [ ]:
def _show_scorecard() -> None:
    results = _analysis()["results"]
    need = _gate(len(results))
    decision = release_decision(results, need)
    print(f"slices searched     {len(results)} (one attribute each, at least {MIN_SLICE} "
          "documents)")
    print(f"size gate           {need} documents (MARGIN {MARGIN}, power {POWER}, "
          f"ALPHA {ALPHA}/{len(results)})")
    print(f"real regressions    {', '.join(decision.blocking) or 'none'}")
    print(f"too small to say    {len(decision.too_small)} slices")
    print(f"decision            {'ship' if decision.ship else 'do not ship'}")


_try("scorecard", _show_scorecard,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5",
            "exercise 6"))

## What you built, and where it goes next

A release decision that a reviewer can check line by line: which slices were searched, how
each was measured, which correction was applied to how many tests, and why a slice the data
cannot judge is named as such rather than quietly passed. Module 8 watches the same slices in
production, where the question becomes whether today's drop is real; module 11 puts this
decision, and the numbers behind it, into the conformity pack.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_find_slices),
                              ("exercise 2", _check_macro_f1),
                              ("exercise 3", _check_bootstrap),
                              ("exercise 4", _check_adjust),
                              ("exercise 5", _check_required),
                              ("exercise 6", _check_decision)):
            _try(_name, _check)
    _progress_board()
    print(f"\nlesson wall time so far: {time.perf_counter() - _LESSON_T0:.1f} s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.